In [1]:
print("Hello World")

Hello World


# Package Import

In [19]:
from typing import List, TypedDict, Literal,Dict
from pydantic import BaseModel, Field
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

In [3]:
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec

c:\Users\TPWODL\miniconda3\envs\genai\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
load_dotenv()

True

# 1. Data Ingestion Layer (Raw Data Entry)

In [5]:
docs = PyPDFLoader(r"C:\Users\TPWODL\New folder_Content\Advanced_RAG_Project\data\raw\Supply Code, 2019 with Gazette.pdf").load()

In [6]:
docs

[Document(metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\data\\raw\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th  August, 2019 \n \nNo.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section \n50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, \n2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity \nRegulatory Commission hereby make the following Sup ply Code by Notification to govern \nsupply of elect

In [7]:
print(len(docs))

95


In [10]:
docs[0]

Document(metadata={'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\data\\raw\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}, page_content='1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th  August, 2019 \n \nNo.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section \n50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, \n2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity \nRegulatory Commission hereby make the following Sup ply Code by Notification to govern \nsupply of electr

In [11]:
# To view the full object content
print(docs[0])

page_content='1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
ODISHA ELECTRICITY REGULATORY COMMISSION 
BHUBANESWAR – 751 021 
***** 
 
NOTIFICATION 
The 27 th  August, 2019 
 
No.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section 
50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, 
2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity 
Regulatory Commission hereby make the following Sup ply Code by Notification to govern 
supply of electricity by the licensee/supplier to t he consumers / end users and measures for 
recovery of electricity charges, intervals for bill ing of electricity charges, disconnection of 
supply of electricity for non-payment thereof, rest oration of supply of electricity, measures for 
preventing, tampering, distress or damage to electr ical plant or electrical line or meter, entry of 
distribution licensee or any person acting on his b ehalf for disconnecting s

In [12]:
# If it's a LangChain Document, view the text content specifically
print(docs[0].page_content)

1 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
ODISHA ELECTRICITY REGULATORY COMMISSION 
BHUBANESWAR – 751 021 
***** 
 
NOTIFICATION 
The 27 th  August, 2019 
 
No.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section 
50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, 
2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity 
Regulatory Commission hereby make the following Sup ply Code by Notification to govern 
supply of electricity by the licensee/supplier to t he consumers / end users and measures for 
recovery of electricity charges, intervals for bill ing of electricity charges, disconnection of 
supply of electricity for non-payment thereof, rest oration of supply of electricity, measures for 
preventing, tampering, distress or damage to electr ical plant or electrical line or meter, entry of 
distribution licensee or any person acting on his b ehalf for disconnecting supply and remo

In [13]:
print(docs[0].metadata)

{'producer': 'GPL Ghostscript 9.06', 'creator': 'PScript5.dll Version 5.2', 'creationdate': '2019-10-19T13:51:32+05:30', 'author': 'spm', 'moddate': '2019-11-06T13:21:17+05:30', 'title': 'Microsoft Word - Supply Code, 2019 Final 19.10.2019', 'source': 'C:\\Users\\TPWODL\\New folder_Content\\Advanced_RAG_Project\\data\\raw\\Supply Code, 2019 with Gazette.pdf', 'total_pages': 95, 'page': 0, 'page_label': '1'}


In [14]:
print(docs[6].page_content)

7 
 
units that would have been consumed had the maximum  demand been maintained 
throughout the same period and is usually expressed as a percentage, that is, 
Load Factor in Percentage = (Actual units consumed during a given period / 
Maximum demand in KW X Number of Hrs during the period) X100, 
‘Load Factor’ in case of loads up to and excluding connected load  of 100KW is 
the ratio of the total number of units consumed dur ing a given period to the total 
number of units that would have been consumed had t he contract demand been 
maintained throughout the same period and is usuall y expressed as a percentage, 
that is, 
Load Factor in Percentage = (Actual units consumed during a given period / 
Contract demand in KW X Number of Hrs during the period) X 100, 
(43) “Low Tension Consumer ” means a consumer who obtains supply from the 
licensee/supplier at low voltage; 
(44) “Main”  means any electric supply-line through which elect ricity is, or is intended 
to be supplied; 
(45) “M

In [15]:
print(docs[23].page_content)

24  
 
Supplementary Agreement 
50. Whenever restriction on power supply is imposed and  power purchased from other States 
or agencies is supplied to the consumer on special request, a supplementary agreement 
shall be executed which shall remain in force for the period of such restriction. 
 Record of Disconnection and Reconnection 
51. The licensee/supplier shall maintain a record of di sconnection and reconnection. The 
licensee/supplier shall intimate in writing the dat e of disconnection including amount 
due, meter reading at the time of disconnection to the consumer within seven days of 
disconnection, and obtain acknowledgement of the co nsumer or his authorized 
representative. 
 The information shall be placed in the website with  intimation to 
consumers.  
 Security Deposit 
52. (i)  Any person entering into an agreement with the  licensee/supplier for supply of 
power shall deposit such amount to cover charges (i .e. demand/fixed charges and 
 energy charges as applicable

In [20]:
import re
import json
import hashlib
from pathlib import Path
from typing import Optional

In [21]:
# -----------------------------
# 1. BASIC TEXT CLEANING
# -----------------------------
def clean_text(raw_text: str) -> str:
    text = raw_text

    # Remove page numbers / isolated numbers
    text = re.sub(r"\n\s*\d+\s*\n", "\n", text)

    # Fix broken words (e.g., "dur ing" -> "during")
    text = re.sub(r"(\w)\s+(\w)", r"\1\2", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Remove unwanted special characters
    text = re.sub(r"[‘’“”]", '"', text)

    # Normalize spacing around commas and brackets
    text = re.sub(r"\s*,\s*", ", ", text)

    return text.strip()


# -----------------------------
# 2. REMOVE HEADER / FOOTER NOISE
# -----------------------------
def remove_noise(text: str) -> str:
    lines = text.split("\n")

    # Remove repeated lines (header/footer detection)
    line_freq = {}
    for line in lines:
        line = line.strip()
        if line:
            line_freq[line] = line_freq.get(line, 0) + 1

    # Remove lines appearing too frequently
    cleaned_lines = [
        line for line in lines
        if line_freq.get(line.strip(), 0) < 3
    ]

    return "\n".join(cleaned_lines)


# -----------------------------
# 3. EXTRACT DEFINITIONS
# -----------------------------
def extract_definitions(text: str) -> List[Dict]:
    pattern = r'\((\d+)\)\s*"(.*?)"\s*means\s*(.*?)(?=\(\d+\)|$)'
    matches = re.findall(pattern, text, re.DOTALL)

    results = []
    for num, title, content in matches:
        results.append({
            "type": "definition",
            "id": int(num),
            "title": title.strip(),
            "content": content.strip()
        })

    return results


# -----------------------------
# 4. EXTRACT FORMULAS
# -----------------------------
def extract_formulas(text: str) -> List[Dict]:
    formulas = []

    # Load Factor formulas
    lf_pattern = r'Load Factor.*?=\s*(.*?100)'
    matches = re.findall(lf_pattern, text)

    for m in matches:
        formulas.append({
            "type": "formula",
            "name": "Load Factor",
            "expression": m.strip()
        })

    # Security deposit formula
    sd_pattern = r'Security Deposit.*?=\s*(.*)'
    match = re.search(sd_pattern, text)
    if match:
        formulas.append({
            "type": "formula",
            "name": "Security Deposit",
            "expression": match.group(1).strip()
        })

    return formulas


# -----------------------------
# 5. EXTRACT TABLE (LOAD FACTOR)
# -----------------------------
def extract_table(text: str) -> List[Dict]:
    table_pattern = r'\d+\.\s*(.*?)\s*(\d+)%'
    matches = re.findall(table_pattern, text)

    table = []
    for category, value in matches:
        table.append({
            "category": category.strip(),
            "load_factor_percent": int(value)
        })

    return table


# -----------------------------
# 6. SECTION SPLITTING
# -----------------------------
def split_sections(text: str) -> List[Dict]:
    sections = re.split(r'\n\s*\d+\.\s+', text)

    structured = []
    for sec in sections:
        sec = sec.strip()
        if len(sec) > 50:
            structured.append({
                "type": "section",
                "content": sec
            })

    return structured


# -----------------------------
# 7. MAIN CLEANING PIPELINE
# -----------------------------
def clean_pdf_data(raw_text: str) -> Dict:
    # Step 1: basic cleaning
    text = clean_text(raw_text)

    # Step 2: remove noise
    text = remove_noise(text)

    # Step 3: structured extraction
    definitions = extract_definitions(text)
    formulas = extract_formulas(text)
    table = extract_table(text)
    sections = split_sections(text)

    return {
        "clean_text": text,
        "definitions": definitions,
        "formulas": formulas,
        "table": table,
        "sections": sections
    }


In [24]:
# Join the text from all pages/documents into one string
raw_text = " ".join([doc.page_content for doc in docs])




In [25]:
raw_text

'1 \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nODISHA ELECTRICITY REGULATORY COMMISSION \nBHUBANESWAR – 751 021 \n***** \n \nNOTIFICATION \nThe 27 th  August, 2019 \n \nNo.OERC/Engg/92/2003(Vol-VII)/1210 - In exercise of  the powers conferred by Section \n50 read with and Section 181 (2) (t), (v), (w) and (x) read with Part-VI of the Electricity Act, \n2003 (Act 36 of 2003) and all other powers enabling  it in this behalf, the Odisha Electricity \nRegulatory Commission hereby make the following Sup ply Code by Notification to govern \nsupply of electricity by the licensee/supplier to t he consumers / end users and measures for \nrecovery of electricity charges, intervals for bill ing of electricity charges, disconnection of \nsupply of electricity for non-payment thereof, rest oration of supply of electricity, measures for \npreventing, tampering, distress or damage to electr ical plant or electrical line or meter, entry of \ndistribution licensee or any person acting on his b ehalf fo

In [28]:
print(type(raw_text))

<class 'str'>


In [29]:
cleaned_output_docs = clean_pdf_data(raw_text)

In [30]:
cleaned_output_docs

{'clean_text': '1ODISHAELECTRICITYREGULATORYCOMMISSIONBHUBANESWAR – 751021 ***** NOTIFICATIONThe27thAugust, 2019No.OERC/Engg/92/2003(Vol-VII)/1210 - InexerciseofthepowersconferredbySection50readwithandSection181 (2) (t), (v), (w) and (x) readwithPart-VIoftheElectricityAct, 2003 (Act36of2003) andallotherpowersenablingitinthisbehalf, theOdishaElectricityRegulatoryCommissionherebymakethefollowingSupplyCodebyNotificationtogovernsupplyofelectricitybythelicensee/suppliertot heconsumers / endusersandmeasuresforrecoveryofelectricitycharges, intervalsforbillingofelectricitycharges, disconnectionofsupplyofelectricityfornon-paymentthereof, restorationofsupplyofelectricity, measuresforpreventing, tampering, distressordamagetoelectricalplantorelectricallineormeter, entryofdistributionlicenseeoranypersonactingonhisb ehalffordisconnectingsupplyandremovingthemeter, entryforreplacing, alteringormaintainingelectriclinesorelectricalplantsormeterandsuchothermatters. OdishaElectricityRegulatoryCommissionDi

In [31]:
print(type(cleaned_output_docs))

<class 'dict'>


In [34]:
print(cleaned_output_docs.keys())

dict_keys(['clean_text', 'definitions', 'formulas', 'table', 'sections'])


In [35]:
print(cleaned_output_docs['clean_text'])

1ODISHAELECTRICITYREGULATORYCOMMISSIONBHUBANESWAR – 751021 ***** NOTIFICATIONThe27thAugust, 2019No.OERC/Engg/92/2003(Vol-VII)/1210 - InexerciseofthepowersconferredbySection50readwithandSection181 (2) (t), (v), (w) and (x) readwithPart-VIoftheElectricityAct, 2003 (Act36of2003) andallotherpowersenablingitinthisbehalf, theOdishaElectricityRegulatoryCommissionherebymakethefollowingSupplyCodebyNotificationtogovernsupplyofelectricitybythelicensee/suppliertot heconsumers / endusersandmeasuresforrecoveryofelectricitycharges, intervalsforbillingofelectricitycharges, disconnectionofsupplyofelectricityfornon-paymentthereof, restorationofsupplyofelectricity, measuresforpreventing, tampering, distressordamagetoelectricalplantorelectricallineormeter, entryofdistributionlicenseeoranypersonactingonhisb ehalffordisconnectingsupplyandremovingthemeter, entryforreplacing, alteringormaintainingelectriclinesorelectricalplantsormeterandsuchothermatters. OdishaElectricityRegulatoryCommissionDistribution (Cond

In [36]:
print(cleaned_output_docs['definitions'])

[{'type': 'definition', 'id': 1, 'title': 'Act', 'content': 'theElectricityAct, 2003 (Act36of2003) ;'}, {'type': 'definition', 'id': 2, 'title': 'Agreement" withitsgrammaticalvariationsandcognateexpressionsmeansanagreemententeredintobythelicensee/supplierandtheconsumerinaccordancetoRegulation-48intheformatatFormno.1- 3oftheseRegulation; (3) "AccreditedTestLaboratory', 'content': 'a testlaboratoryaccreditedbyNationalAccreditationBoardforTestingandCalibrationLaboratories (NABL); listofwhichmaybeprominentlydisplayedinthelicensee/ supplier"sfieldofficepremisesfortheknowledgeoftheconsumers.'}, {'type': 'definition', 'id': 4, 'title': 'Ampere', 'content': 'a unitofelectriccurrentandistheunvaryingelectriccurrentwhichwhenpassedthrougha solutionofnitrateofsilverinwater, inaccordancewiththespecificationsetoutinAnnexure-IoftheIndianElectricityRules, 1956orRules/RegulationsmadeunderSection53oftheA ct, depositssilverattherateof0.001118ofa grammepersecond; theaforesaidunitisequivalenttothecurrentwhi

In [37]:
print(cleaned_output_docs['formulas'])

[]


In [38]:
print(cleaned_output_docs['table'])

[{'category': 'SHORTTITLE, COMMENCEMENT1.1ThisCodeshallbecalledthe "OdishaElectricityRegulatoryCommissionDistribution (ConditionsofSupply) Code, 2019; hereinafterreferredtoas "Code". 1.2ThisCodeshallcomeintoforceonthedateofpublicationintheOfficialGazette. 1.3ThisCodeshallextendtothewholeoftheStateofOdisha. 21.4ThisCodeshallbeapplicableto: (a) allDistributionandRetailSupplylicensee/suppliersincludingDeemedlicensee/suppliers, allconsumers, endusersofelectricityintheStateofOdisha; (b) allotherpersonsdulydoingthebusinessofdistribution/supplyofelectricityinhisareaofsupply; (c) allotherpersonswhoareexemptedunderSection13oftheAct; and (d) unauthorizedsupply, unauthorizeduse, diversionandothermeansofunauthorizeduse/ abstractionofelectricity. 1.5TheOdishaGeneralClausesAct, 1937shallapplytotheinterpretationofthisCode. 1.6ThisCodeshallsupersedetheOdishaElectricityRegulatoryCommissionDistribution (ConditionsofSupply) Code, 2004anditssubsequentamendments. 1.7Ontheapplicationofthelicensee/supplier(s

In [39]:
print(cleaned_output_docs['sections'])

[{'type': 'section', 'content': '1ODISHAELECTRICITYREGULATORYCOMMISSIONBHUBANESWAR – 751021 ***** NOTIFICATIONThe27thAugust, 2019No.OERC/Engg/92/2003(Vol-VII)/1210 - InexerciseofthepowersconferredbySection50readwithandSection181 (2) (t), (v), (w) and (x) readwithPart-VIoftheElectricityAct, 2003 (Act36of2003) andallotherpowersenablingitinthisbehalf, theOdishaElectricityRegulatoryCommissionherebymakethefollowingSupplyCodebyNotificationtogovernsupplyofelectricitybythelicensee/suppliertot heconsumers / endusersandmeasuresforrecoveryofelectricitycharges, intervalsforbillingofelectricitycharges, disconnectionofsupplyofelectricityfornon-paymentthereof, restorationofsupplyofelectricity, measuresforpreventing, tampering, distressordamagetoelectricalplantorelectricallineormeter, entryofdistributionlicenseeoranypersonactingonhisb ehalffordisconnectingsupplyandremovingthemeter, entryforreplacing, alteringormaintainingelectriclinesorelectricalplantsormeterandsuchothermatters. OdishaElectricityRegul

In [45]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Convert strings into Document objects
docs = [Document(page_content=text, metadata={}) for text in cleaned_output_docs]




In [52]:
docs

[Document(metadata={}, page_content='clean_text'),
 Document(metadata={}, page_content='definitions'),
 Document(metadata={}, page_content='formulas'),
 Document(metadata={}, page_content='table'),
 Document(metadata={}, page_content='sections')]

In [53]:

chunks = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30
).split_documents(docs)

In [55]:
print(len(chunks))

5
